# Classical CV, GCV, AICc and BIC

Cortés-Toto, Guerrero & Reyes (2017) compare these *in-sample* selection rules. Their numerical scales differ, so compare the selected λ or normalized smoothness rather than raw criterion magnitudes.

In [ ]:
import awesomefigures as af
from matplotlib_inline.backend_inline import set_matplotlib_formats

# Publication styling only; numerical estimates and metrics are unchanged.
af.set_style("elegante", font="DejaVu Sans", use_tex=False,
             palette="okabe-ito", rasterize=False)
set_matplotlib_formats("svg")


## Create published-style data

We generate a deterministic nonlinear trend plus Gaussian noise; we do not use future observations or the experimental forecasting TSCV objective.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from trendestimation import TrendSmoother,simulate_cortes_toto,select_lambda
data=simulate_cortes_toto(n=80,shape="nonlinear",noise_sd=.5,seed=11)
y=data.observed
fig,ax=plt.subplots(figsize=(10,3.5))
ax.plot(y,color="0.72",label="observed");ax.plot(data.trend,label="latent trend")
ax.legend();ax.set(title="Controlled simulation with known trend");plt.show()

## Fit every criterion

For fixed μ=0, classical leverage-based CV, trace-based GCV, corrected AIC and BIC can be evaluated using the reused spectral decomposition.

In [ ]:
sm=TrendSmoother(len(y),order=2)
names=["cv","gcv","aicc","bic"]
results={name:select_lambda(sm,y,criterion=name,mu=0.0) for name in names}
for name,r in results.items():
    print(f"{name.upper():5s}: λ={r.lambda_:.6g}, normalized S={r.smoothness:.3f}, score={r.score:.4g}")
fig,axs=plt.subplots(2,2,figsize=(11,7))
for ax,(name,r) in zip(axs.flat,results.items()):
    ok=np.isfinite(r.grid)&np.isfinite(r.scores)&(r.grid>0)
    ax.semilogx(r.grid[ok],r.scores[ok])
    if np.isfinite(r.lambda_):ax.axvline(r.lambda_,color="0.3",ls=":")
    ax.set(xlabel="λ",ylabel=name.upper(),title=f"{name.upper()} selection curve")
fig.tight_layout();plt.show()

## Recovered trends and diagnostic loss

The known synthetic truth provides descriptive *latent-trend recovery error*, not out-of-sample predictive accuracy.

In [ ]:
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(y,color="0.85",label="noisy series")
ax.plot(data.trend,"k--",label="true trend")
for name,r in results.items():
    fit=sm.fit(y,lambda_=r.lambda_)
    rmse=np.sqrt(np.mean((fit.trend-data.trend)**2))
    ax.plot(fit.trend,label=f"{name.upper()}; latent RMSE={rmse:.3f}")
ax.legend(ncol=2);ax.set(title="Criterion-dependent trend recovery");plt.show()

## Gaussian MLE and its two variance components

The four selectors above are the **classical in-sample criteria**. A separate Gaussian marginal maximum-likelihood selector, `criterion="mle"`, assumes independent observation noise and a Gaussian penalty for the latent second differences. It estimates a smoothing penalty from the profile likelihood, then reports an **observation-noise variance** and a **latent difference variance**. These are **not** the residual MSE or a forecast error variance. In this example `mu=0.0` is fixed.

In [ ]:
from trendestimation import estimate_variance_components

mle = select_lambda(sm, y, criterion="mle", mu=0.0)
components = estimate_variance_components(
    sm, y, lambda_=mle.lambda_, mu=0.0
)
print(f"Gaussian MLE: lambda={mle.lambda_:.5g}")
print(f"Observation-noise variance: {components.observation_variance:.5g}")
print(f"Latent second-difference variance: {components.difference_variance:.5g}")
print(f"Observation SD: {components.observation_sd:.5g}")
print(f"Latent difference SD: {components.difference_sd:.5g}")
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(y, ".", color="0.7", label="Synthetic data")
ax.plot(data.trend, "k--", label="Known synthetic trend")
ax.plot(sm.fit(y, lambda_=mle.lambda_, mu=0.0).trend, label="Gaussian MLE trend")
ax.set(xlabel="Observation", ylabel="Value", title="Profile-likelihood smoothing")
ax.legend()
plt.show()